In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/icml_face_data.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/fer2013.tar.gz
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/example_submission.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/test.csv
/kaggle/input/models/ldavi22/vgg4-r5/pytorch/default/1/vgg4_r5.pt


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# ── Model ─────────────────────────────────────────────────────────────────────

class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1a = nn.Conv2d(1,   64,  3, padding=1)
        self.bn1a   = nn.BatchNorm2d(64)
        self.conv1b = nn.Conv2d(64,  64,  3, padding=1)
        self.bn1b   = nn.BatchNorm2d(64)

        self.conv2a = nn.Conv2d(64,  128, 3, padding=1)
        self.bn2a   = nn.BatchNorm2d(128)
        self.conv2b = nn.Conv2d(128, 128, 3, padding=1)
        self.bn2b   = nn.BatchNorm2d(128)

        self.conv3a = nn.Conv2d(128, 256, 3, padding=1)
        self.bn3a   = nn.BatchNorm2d(256)
        self.conv3b = nn.Conv2d(256, 256, 3, padding=1)
        self.bn3b   = nn.BatchNorm2d(256)

        self.conv4a = nn.Conv2d(256, 512, 3, padding=1)
        self.bn4a   = nn.BatchNorm2d(512)
        self.conv4b = nn.Conv2d(512, 512, 3, padding=1)
        self.bn4b   = nn.BatchNorm2d(512)

        self.relu      = nn.ReLU()
        self.pool      = nn.MaxPool2d(2, 2)
        self.dropout2d = nn.Dropout2d(0.2)
        self.dropout   = nn.Dropout(0.5)
        self.flatten   = nn.Flatten()

        self.fc1    = nn.Linear(512 * 3 * 3, 512)
        self.output = nn.Linear(512, 7)

    def forward(self, x):
        x = self.relu(self.bn1a(self.conv1a(x)))
        x = self.relu(self.bn1b(self.conv1b(x)))
        x = self.pool(x)
        x = self.dropout2d(x)

        x = self.relu(self.bn2a(self.conv2a(x)))
        x = self.relu(self.bn2b(self.conv2b(x)))
        x = self.pool(x)
        x = self.dropout2d(x)

        x = self.relu(self.bn3a(self.conv3a(x)))
        x = self.relu(self.bn3b(self.conv3b(x)))
        x = self.pool(x)
        x = self.dropout2d(x)

        x = self.relu(self.bn4a(self.conv4a(x)))
        x = self.relu(self.bn4b(self.conv4b(x)))
        x = self.pool(x)
        x = self.dropout2d(x)

        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.output(x)
        return x

# ── Dataset ───────────────────────────────────────────────────────────────────

class TestDataset(Dataset):
    def __init__(self, df):
        self.pixels = torch.tensor(
            np.stack([
                np.array(p.split(), dtype=np.float32).reshape(1, 48, 48) / 255.0
                for p in df['pixels'].values
            ])
        )

    def __len__(self):
        return len(self.pixels)

    def __getitem__(self, idx):
        return self.pixels[idx]

# ── Load data + model ─────────────────────────────────────────────────────────

df = pd.read_csv("/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/test.csv")

model = Net().to(device)
model.load_state_dict(torch.load("/kaggle/input/models/ldavi22/vgg4-r5/pytorch/default/1/vgg4_r5.pt", map_location=device))
model.eval()

# ── Inference ─────────────────────────────────────────────────────────────────

test_dataset = TestDataset(df)
test_loader  = DataLoader(test_dataset, batch_size=256, shuffle=False)

predictions = []

with torch.no_grad():
    for inputs in test_loader:
        inputs = inputs.to(device)
        output = model(inputs)
        preds  = output.argmax(dim=1).cpu().numpy()
        predictions.extend(preds)

# ── Submission ────────────────────────────────────────────────────────────────

submission = pd.DataFrame({
    "id":      range(len(predictions)),
    "emotion": predictions
})

submission.to_csv("submission.csv", index=False)
print(submission.head())
print(f"Submission shape: {submission.shape}")

   id  emotion
0   0        6
1   1        1
2   2        0
3   3        6
4   4        3
Submission shape: (7178, 2)


In [3]:
!ls

submission.csv
